# Cross-cell C2 · 冻结表征与下游表现

在服务器用 `representation-inputs.json` 的 `results` 指定五个原 checkpoint 的表征回执目录。各模型使用相同的 128 个测试条件。依赖表达输入的读出取每条件相同顺序的 12 个冻结对照，PerturbDiff 的读出是固定 K562 上下文的扰动条件表示。表中记录每个模型截取的层。各层用途不同，归一化有效秩和几何一致性描述各层，模型间趋势只作探索。纯秩升高不足以证明表征更好。

谱使用中心化特征的协方差能量。几何一致性通过同一扰动的潜空间邻居与实际响应邻居的 Spearman 相关刻画。额外输出每个条件的几何一致性与五项冻结指标的关联。模型许多设计不同，检验 token 的因果影响使用同架构配对消融。

In [ ]:

from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

root = Path.cwd()
plan = json.loads((root / "representation-inputs.json").read_text())["results"]
metrics = ["txpert_macro_pearson_delta_all", "trishift_pearson_delta_all", "systema_pearson_all", "mean_mse", "retrieval"]
metric_titles = ["TxPert Pearson", "TriShift Pearson", "Systema Pearson", "MSE (lower)", "Retrieval"]
identities, names, reference = set(), None, None
audit, distributions, associations = [], {}, []

def cosine_matrix(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    assert (norms[:,0] > 0).all()
    unit = vectors / norms
    return unit @ unit.T

for name, directory in plan.items():
    path = root / directory
    r = json.loads((path / "result.json").read_text())
    assert r["status"] == "representations_complete" and r["test_updates"] == 0
    identities.add(tuple(r[k] for k in ["cohort_sha256", "paired_rows_sha256", "score_panel_sha256", "anchor_rows_sha256"]))
    matrices = path / "condition-representations.npz"
    assert hashlib.sha256(matrices.read_bytes()).hexdigest() == r["representation_sha256"]
    with np.load(matrices, allow_pickle=False) as z:
        labels, v, actual = z["condition_ids"], z["representations"].astype(float), z["observed_delta"]
    assert len(set(labels)) == len(labels) == len(v) == 128
    if names is None:
        names, reference = labels, actual
    else:
        np.testing.assert_array_equal(names, labels)
        np.testing.assert_allclose(reference, actual, rtol=1e-7, atol=1e-10)
    performance_path = Path(r["performance_receipt"])
    assert hashlib.sha256(performance_path.read_bytes()).hexdigest() == r["performance_receipt_sha256"]
    performance = json.loads(performance_path.read_text())
    raw = (performance_path.parent / "conditions.jsonl").read_bytes()
    assert hashlib.sha256(raw).hexdigest() == performance["conditions_file_sha256"]
    rows = [json.loads(line) for line in raw.splitlines()]
    assert [x["condition_id"] for x in rows] == labels.tolist()
    per_condition = pd.DataFrame([{**{m["metric_id"]:m["value"] for m in row["results"]},
                                  "mean_mse":row["mean_mse"], "retrieval":row["retrieval"]} for row in rows])
    center = v - v.mean(0)
    values = np.linalg.svd(center, compute_uv=False) ** 2
    energy = values / values.sum()
    positive = energy[energy > 0]
    rank = np.exp(-np.sum(positive * np.log(positive)))
    state_similarity = cosine_matrix(center)
    observed_similarity = cosine_matrix(actual)
    keep = ~np.eye(len(v), dtype=bool)
    alignment = [
        spearmanr(state_similarity[i, mask], observed_similarity[i, mask]).statistic
        for i, mask in enumerate(keep)
    ]
    rank_max = min(len(v) - 1, v.shape[1])
    pair_angles = state_similarity[np.triu(keep)]
    distributions[name] = {"energy": np.cumsum(energy), "cosine":pair_angles, "alignment":alignment}
    for key in metrics:
        corr = spearmanr(alignment, per_condition[key], nan_policy="omit").statistic
        associations.append(dict(model=name, metric=key, spearman=corr))
    audit.append(dict(model=name, layer=r["analysis_layer"], conditions=len(v), width=v.shape[1],
                      effective_rank=rank, normalized_rank=rank / rank_max,
                      centered_variance_ratio=values.sum()/np.square(v).sum(),
                      median_cosine=np.median(pair_angles), mean_geometry_alignment=np.nanmean(alignment),
                      retrieval=performance["retrieval_mean"]))
assert len(identities) == 1 and len(audit) == 5
table = pd.DataFrame(audit)
table.to_csv(root / "representation-summaries.csv", index=False)
association = pd.DataFrame(associations)
association.to_csv(root / "representations-vs-performance.csv", index=False)
display(table.round(5))
display(association.pivot(index="model",columns="metric",values="spearman").round(5))

fig, axes = plt.subplots(1, 3, figsize=(14.3, 4.1), layout="constrained")
for name, vals in distributions.items():
    axes[0].plot(np.arange(1, len(vals["energy"])+1), vals["energy"], label=name)
    axes[1].hist(vals["cosine"], bins=22, histtype="step", density=True, label=name)
axes[0].set(title="Centered energy spectrum", xlabel="Sorted component")
axes[1].set(title="Pairwise centered cosine", xlabel="Cosine")
axes[2].boxplot([distributions[m]["alignment"] for m in table.model], tick_labels=table.model)
axes[2].set(title="Condition-neighborhood alignment", ylabel="Spearman ρ with observed response geometry")
axes[2].tick_params(axis="x", rotation=25)
axes[0].legend()
fig.savefig(root / "representation-geometry.png", dpi=170)
plt.show()

heat = association.pivot(index="model",columns="metric",values="spearman").reindex(table.model)[metrics]
fig, ax = plt.subplots(figsize=(10.8, 4.9), layout="constrained")
img = ax.imshow(heat.values, cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
ax.set(xticks=range(len(metrics)), xticklabels=metric_titles, yticks=range(len(table)), yticklabels=table.model,
       title="Within-model association with response geometry")
for y, row in enumerate(heat.values):
    for x, val in enumerate(row):
        ax.text(x, y, f"{val:.2f}", ha="center", va="center")
fig.colorbar(img, ax=ax, label="Spearman ρ across perturbations")
fig.savefig(root / "geometry-performance-association.png", dpi=170)
plt.show()

cols = ["effective_rank","normalized_rank","centered_variance_ratio","median_cosine","mean_geometry_alignment","retrieval"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7.5), layout="constrained")
for ax, col in zip(axes.flat, cols, strict=True):
    bars = ax.bar(table.model, table[col], color="#285d9f")
    ax.bar_label(bars, fmt="%.3g")
    ax.set_title(col.replace("_", " "))
    ax.tick_params(axis="x", rotation=23)
fig.savefig(root / "representation-summary-metrics.png", dpi=170)
plt.show()
